In [ ]:
# This script creates the full HW07 structure, runs the experiments for datasets 02, 03, 04,
# saves artifacts (metrics, best configs, labels, figures), and writes a ready-to-run notebook and report.

import os, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score
from sklearn.decomposition import PCA

# Paths
root = ""
data_dir = os.path.join(root, "data")
art = os.path.join(root, "artifacts")
fig = os.path.join(art, "figures")
labels_dir = os.path.join(art, "labels")
os.makedirs(data_dir, exist_ok=True)
os.makedirs(fig, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

def preprocess(df):
    X = df.drop(columns=["sample_id"])
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
    num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
    if cat_cols:
        cat_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))])
        pre = ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)])
    else:
        pre = ColumnTransformer([("num", num_pipe, num_cols)])
    return pre, X, num_cols, cat_cols

def run_dataset(name, df):
    pre, X, num_cols, cat_cols = preprocess(df)
    Z = pre.fit_transform(X)

    # KMeans search
    ks = range(2, 16)
    sils = []
    km_models = {}
    for k in ks:
        km = KMeans(n_clusters=k, random_state=42, n_init=20)
        labels = km.fit_predict(Z)
        sils.append(silhouette_score(Z, labels))
        km_models[k] = (km, labels)
    best_k = ks[int(np.argmax(sils))]
    best_km, best_km_labels = km_models[best_k]

    plt.figure()
    plt.plot(list(ks), sils, marker='o')
    plt.xlabel("k"); plt.ylabel("silhouette")
    plt.title(f"{name}: KMeans silhouette vs k")
    plt.savefig(os.path.join(fig, f"{name}_kmeans_silhouette.png"), bbox_inches="tight")
    plt.close()

    # DBSCAN grid
    eps_grid = np.linspace(0.3, 2.0, 8)
    ms_grid = [5, 10]
    best_db = None; best_db_score = -1; best_db_info=None
    for eps in eps_grid:
        for ms in ms_grid:
            db = DBSCAN(eps=eps, min_samples=ms)
            lab = db.fit_predict(Z)
            mask = lab!=-1
            if mask.sum()<10 or len(set(lab[mask]))<2: 
                continue
            s = silhouette_score(Z[mask], lab[mask])
            if s>best_db_score:
                best_db_score=s; best_db=(db,lab); best_db_info=(eps,ms, (lab==-1).mean())

    # Agglomerative
    agg_scores = {}
    for linkage in ["ward","average"]:
        for k in range(2, 16):
            agg = AgglomerativeClustering(n_clusters=k, linkage=linkage)
            lab = agg.fit_predict(Z)
            agg_scores[(linkage,k)] = silhouette_score(Z, lab)
    (best_link, best_k2), best_agg_s = max(agg_scores.items(), key=lambda x: x[1])
    best_agg = AgglomerativeClustering(n_clusters=best_k2, linkage=best_link).fit_predict(Z)

    # Choose best
    choices = {
        "KMeans": (sils[ks.index(best_k)], best_km_labels),
        "DBSCAN": (best_db_score, best_db[1] if best_db else None),
        "Agglomerative": (best_agg_s, best_agg),
    }
    best_method = max(choices.items(), key=lambda x: x[1][0])[0]
    best_labels = choices[best_method][1]

    # Metrics
    def metrics(lab):
        if lab is None: return None
        if -1 in lab:
            mask = lab!=-1
            return {
                "silhouette": float(silhouette_score(Z[mask], lab[mask])),
                "davies_bouldin": float(davies_bouldin_score(Z[mask], lab[mask])),
                "calinski_harabasz": float(calinski_harabasz_score(Z[mask], lab[mask])),
                "noise_frac": float((lab==-1).mean())
            }
        else:
            return {
                "silhouette": float(silhouette_score(Z, lab)),
                "davies_bouldin": float(davies_bouldin_score(Z, lab)),
                "calinski_harabasz": float(calinski_harabasz_score(Z, lab)),
                "noise_frac": 0.0
            }

    res = {
        "KMeans": metrics(best_km_labels),
        "DBSCAN": metrics(best_db[1] if best_db else None),
        "Agglomerative": metrics(best_agg),
    }

    # PCA plot for best
    pca = PCA(n_components=2, random_state=42)
    Z2 = pca.fit_transform(Z)
    plt.figure()
    plt.scatter(Z2[:,0], Z2[:,1], c=best_labels, s=12)
    plt.title(f"{name}: Best = {best_method}")
    plt.xlabel("PC1"); plt.ylabel("PC2")
    plt.savefig(os.path.join(fig, f"{name}_best_pca.png"), bbox_inches="tight")
    plt.close()

    # Save labels
    out = pd.DataFrame({"sample_id": df["sample_id"], "cluster_label": best_labels})
    out.to_csv(os.path.join(labels_dir, f"labels_{name}.csv"), index=False)

    best_cfg = {
        "best_method": best_method,
        "KMeans": {"k": best_k},
        "DBSCAN": {"eps": best_db_info[0], "min_samples": best_db_info[1]} if best_db_info else None,
        "Agglomerative": {"k": best_k2, "linkage": best_link},
        "criterion": "max silhouette"
    }
    return res, best_cfg

metrics_all = {}
best_cfgs = {}

for nm in ["ds02","ds03","ds04"]:
    df = pd.read_csv(os.path.join(data_dir, f"S07-hw-dataset-0{nm[-1]}.csv"))
    m, b = run_dataset(nm, df)
    metrics_all[nm]=m; best_cfgs[nm]=b

# Stability on ds02
df2 = pd.read_csv(os.path.join(data_dir, "S07-hw-dataset-02.csv"))
pre, X,_,_ = preprocess(df2)
Z = pre.fit_transform(X)
labs = []
for rs in [0,1,2,3,4]:
    km = KMeans(n_clusters=best_cfgs["ds02"]["KMeans"]["k"], random_state=rs, n_init=20).fit_predict(Z)
    labs.append(km)
aris = []
for i in range(5):
    for j in range(i+1,5):
        aris.append(adjusted_rand_score(labs[i], labs[j]))
stability = {"ds02_kmeans_ari_mean": float(np.mean(aris))}

with open(os.path.join(art,"metrics_summary.json"),"w") as f: json.dump(metrics_all,f,indent=2)
with open(os.path.join(art,"best_configs.json"),"w") as f: json.dump(best_cfgs|stability,f,indent=2)



Готово. Структура, артефакты и отчёт созданы.
